In [7]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    # base_url="",
)

### What is RAG - Why LLMs need context, the RAG architecture

In [5]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [6]:
llm("Hey, what's up?")

'Not much—just here and ready to help. What’s going on?'

In [7]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Possibly — it depends on the course’s enrollment rules and whether the class is still open.

If you want, I can help you figure it out. Usually the key questions are:
- Is the course still accepting new students?
- Has the registration deadline passed?
- Is there a waitlist or late-add option?
- Do you need instructor permission?

If you’d like, send me the course name or the platform/school, and I can help you draft a quick message asking whether you can still join.


In [8]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [9]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

### The Course FAQ Dataset - Fetching and exploring the FAQ data

In [10]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
courses_raw[:3]

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41}]

In [16]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [20]:
documents[:2]

[{'id': '9e508f2212',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: When does the course start?',
  'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."},
 {'id': 'bfafa427b3',
  'course': 'data-engineering-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: What are the prerequisites for this course?',
  'answer': "To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful 

### Search basics

In [22]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [24]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [25]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [26]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [27]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [29]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [32]:
search_results = search("How do I submit the homeworks?")
print(search_results)

[{'id': 'd5fc98925d', 'course': 'llm-zoomcamp', 'section': 'Capstone Project', 'question': 'Do we submit 2 projects, what does attempt 1 and 2 mean?', 'answer': 'You only need to submit one project. If the submission at the first attempt fails, you can improve it and re-submit during the attempt#2 submission window.\n\n- If you want to submit two projects for the experience and exposure, you must use different datasets and problem statements.\n- If you can’t make it to the attempt#1 submission window, you still have time to catch up to meet the attempt#2 submission window.\n\nRemember that the submission does not count towards the certification if you do not participate in the peer-review of three peers in your cohort.'}, {'id': 'e394e6f738', 'course': 'llm-zoomcamp', 'section': 'Workshop: Open-Source Data Ingestion (dlt)', 'question': 'How do I know which tables are in the db?', 'answer': 'You can use the `db.table_names()` method to list all the tables in the database.'}, {'id': '049

### Building the Prompt

In [33]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [34]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [35]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [38]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [39]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
Capstone Project
Q: Do we submit 2 projects, what does attempt 1 and 2 mean?
A: You only need to submit one project. If the submission at the first attempt fails, you can improve it and re-submit during the attempt#2 submission window.

- If you want to submit two projects for the experience and exposure, you must use different datasets and problem statements.
- If you can’t make it to the attempt#1 submission window, you still have time to catch up to meet the attempt#2 submission window.

Remember that the submission does not count towards the certification if you do not participate in the peer-review of three peers in your cohort.

Workshop: Open-Source Data Ingestion (dlt)
Q: How do I know which tables are in the db?
A: You can use the `db.table_names()` method to list all the tables in the database.

General Course-Related Questions
Q: How should I start the course and follow the weekly workflow?
A: Start with the [

### The LLM

In [40]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

In [52]:
response.output[0].content[0].text

'Yes — you can join now.\n\nThe course materials are available anytime, and the usual advice is to start with the docs, videos, and GitHub repo, then follow the weekly workflow at your own pace. If you’ve joined late, you can still catch up using the course platform deadlines and materials.\n\nIf you want, I can also help you figure out the best way to catch up quickly.'

In [53]:
response.output_text

'Yes — you can join now.\n\nThe course materials are available anytime, and the usual advice is to start with the docs, videos, and GitHub repo, then follow the weekly workflow at your own pace. If you’ve joined late, you can still catch up using the course platform deadlines and materials.\n\nIf you want, I can also help you figure out the best way to catch up quickly.'

In [54]:
response.usage

ResponseUsage(input_tokens=915, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=84, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=999)

In [55]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00106425

In [57]:
message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

In [59]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=message_history
)

In [60]:
response.output_text

'Yes — you can start whenever you want. The videos, GitHub materials, and deadlines are available in the course management platform.'

In [61]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
    
    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    return response.output_text

In [62]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model)
    return answer

In [63]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join and start learning now.

If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [64]:
print(rag("Where can I ask questions with other learners taking the course?"))

You can ask technical questions with other learners in the course **Slack channel**.


In [65]:
print(rag("How do I get a certificate?"))

You can get a certificate only if you finish the course with a live cohort and pass the Capstone project.

Self-paced mode does not include a certificate, because certificates require peer-reviewing 3 capstones during the live course period.


### RAG Helper

In [66]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from openai import OpenAI

documents = load_faq_data()
index = build_index(documents)

openai_client = OpenAI()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.


In [67]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=custom_instructions,
)

In [68]:
assistant.rag("How do I get a certificate?")

'To get a certificate, you need to finish the course with a **live cohort** and **pass the Capstone project**.\n\nCertificates are **not awarded for self-paced mode**, because you need to **peer-review 3 capstones** after submitting your project, and that can only be done while the course is running. If you missed the first homework, that’s okay—homework is **not mandatory** for the certificate.'

In [69]:
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join after the course has started. You can start whenever you want, but if you want a certificate, you need to finish the course with a live cohort and submit your project while submissions are still being accepted.'

### Data Ingestion

In [ ]:
# !uv add sqlitesearch

### Quick RAG Revision (Optional)

In [72]:
assistant.rag("Which teams won the World Cup 2026 on the first day?")

'I don’t see any information in the provided context about the World Cup 2026 or any teams winning it on the first day.'

In [73]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system.\n   - macOS: download the `.pkg` and install it\n   - Windows: download the `.msi` and install it\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. In a terminal, start a model locally with:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To test that the local Ollama server is running, use:\n   ```bash\n   curl http://localhost:11434\n   ```\n\n4. If you want to use it from Python, install the client with:\n   ```bash\n   pip install ollama\n   ```\n\n   Then call it like this:\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": your_prompt}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```'

In [74]:
assistant.rag("How do I run Olama locally?")

'The FAQ context doesn’t mention **Olama/Ollama** specifically.\n\nIt does say you can run the course **locally** instead of Codespaces if you’re comfortable setting up the needed tools, including **Python, `uv`, Jupyter, Docker, and any other module-specific tools**. If you do run locally, you should **document your setup and keep it reproducible**.\n\nIf you meant a specific local setup for Ollama, I’d need more context from the course materials.'

### Function Calling

In [79]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment policy and whether it’s still open.\n\nIf you want, I can help you figure it out quickly. Send me:\n- the course name\n- the school/platform\n- when it starts\n- whether you’re already signed up anywhere\n\nOr, if you’re asking in general, check:\n1. **Enrollment deadline** — some courses allow late registration.\n2. **Prerequisites** — you may need prior approval or prior coursework.\n3. **Seat availability** — the class may be full.\n4. **Who to contact** — instructor, registrar, or course admin.\n\nIf you want, I can also help you draft a short message asking to join.'

In [80]:
response.output

[ResponseOutputMessage(id='msg_0d332f9ca885ae0b006a37ca053e70819996938032f10c3c27', content=[ResponseOutputText(annotations=[], text='Maybe — it depends on the course’s enrollment policy and whether it’s still open.\n\nIf you want, I can help you figure it out quickly. Send me:\n- the course name\n- the school/platform\n- when it starts\n- whether you’re already signed up anywhere\n\nOr, if you’re asking in general, check:\n1. **Enrollment deadline** — some courses allow late registration.\n2. **Prerequisites** — you may need prior approval or prior coursework.\n3. **Seat availability** — the class may be full.\n4. **Who to contact** — instructor, registrar, or course admin.\n\nIf you want, I can also help you draft a short message asking to join.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [8]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [86]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered enroll eligibility can I join after course started"}', call_id='call_YpQTEgzRcehAbAtYQhQ1Fm0k', name='search', type='function_call', id='fc_06703109cdad7f8b006a37cb81fb508199be99f6557029dd8a', namespace=None, status='completed')]

In [87]:
import json

call = response.output[0]
args = json.loads(call.arguments)
args

{'query': 'join course discovered enroll eligibility can I join after course started'}

In [88]:
results = search(**args)
result_json = json.dumps(results, indent=2)

In [92]:
results[:2]

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'}]

In [94]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [95]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered enroll eligibility can I join after course started"}', call_id='call_YpQTEgzRcehAbAtYQhQ1Fm0k', name='search', type='function_call', id='fc_06703109cdad7f8b006a37cb81fb508199be99f6557029dd8a', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_YpQTEgzRcehAbAtYQhQ1Fm0k',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get 

In [96]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course anytime and start learning.\n\nIf you want a certificate, though, you need to submit your project while submissions are still open and complete the course during the live cohort.'

In [97]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(772, 45)

In [ ]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(usage.input_tokens, usage.output_tokens)
print("Total cost: $", round(result["total_cost"], 8))

### The Agentic Loop

In [12]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [13]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [101]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration late join"}
function_call: search {"query":"course FAQ enrollment join discovered course can I still enroll"}


In [103]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late registration FAQ"}
function_call: search {"query":"new student join course after start enroll FAQ"}
function_call: search {"query":"course access enrollment deadline join after discovering course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

A couple of notes:
- You can start learning anytime, and the materials are available.
- If you want a certificate, you need to submit your project while submissions are still open.

If you want, I can also help you with how to start the course or what the workflow looks like. Any other areas you want to explore?


In [14]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [107]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama local run install start model localhost FAQ"}
function_call: search {"query":"run Ollama locally install command ollama serve FAQ"}
iteration #2...
ASSISTANT:
To run Ollama locally:

1. **Install Ollama**
   - **macOS**: download and install the `.pkg` from https://ollama.com/download
   - **Windows**: download and install the `.msi`
   - **Linux**:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a local model**
   ```bash
   ollama run llama3
   ```
   This downloads the model if needed and starts a local chat session.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   You should get a response from the Ollama server.

4. **If you need to restart the server**
   ```bash
   nohup ollama serve > nohup.out 2>&1 &
   ```
   This is useful if you get a connection refused error.

5. **Optional: use it from Python**
   ```bash
   pip install ollama
   ``

'To run Ollama locally:\n\n1. **Install Ollama**\n   - **macOS**: download and install the `.pkg` from https://ollama.com/download\n   - **Windows**: download and install the `.msi`\n   - **Linux**:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a local model**\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the model if needed and starts a local chat session.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response from the Ollama server.\n\n4. **If you need to restart the server**\n   ```bash\n   nohup ollama serve > nohup.out 2>&1 &\n   ```\n   This is useful if you get a connection refused error.\n\n5. **Optional: use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n   print(response[\'messag

In [108]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late discovered course can I still join enroll late FAQ"}
iteration #2...
ASSISTANT:
Yes, you can still join the course.  

If you want to receive a certificate, though, you need to submit your project while submissions are still being accepted.

If you'd like, I can also help clarify how the certificate process or weekly workflow works. Are there other areas you want to explore?


"Yes, you can still join the course.  \n\nIf you want to receive a certificate, though, you need to submit your project while submissions are still being accepted.\n\nIf you'd like, I can also help clarify how the certificate process or weekly workflow works. Are there other areas you want to explore?"

In [109]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course discovered course can I join enrollment late join FAQ"}
iteration #2...
function_call: search {"query":"certificate project while we're still accepting submissions live cohort self-paced mode join course FAQ registration accepted start learning submitting homework while the form is open"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure you submit your project while submissions are still open. Also, certificates are only available if you complete the course with the live cohort, not in self-paced mode.

If you'd like, I can also help you find where to start or explain the certificate requirements in more detail.


"Yes — you can still join the course.\n\nIf you want a certificate, make sure you submit your project while submissions are still open. Also, certificates are only available if you complete the course with the live cohort, not in self-paced mode.\n\nIf you'd like, I can also help you find where to start or explain the certificate requirements in more detail."

In [110]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit what is Queen's Gambit"}
iteration #2...
function_call: search {"query":"queen gambit chess opening queen's gambit opening definition"}
iteration #3...
ASSISTANT:
A **Queen’s Gambit** is a classic **chess opening**. It starts with:

1. **d4 d5**
2. **c4**

White offers the c-pawn as a gambit to try to distract Black’s d-pawn and gain control of the center. It’s one of the oldest and most respected openings in chess.

There are two main versions:
- **Queen’s Gambit Accepted**: Black takes the c-pawn.
- **Queen’s Gambit Declined**: Black doesn’t take it.

If you want, I can also explain the idea behind it in simple terms or show a few common move lines. Anything else you want to explore?


'A **Queen’s Gambit** is a classic **chess opening**. It starts with:\n\n1. **d4 d5**\n2. **c4**\n\nWhite offers the c-pawn as a gambit to try to distract Black’s d-pawn and gain control of the center. It’s one of the oldest and most respected openings in chess.\n\nThere are two main versions:\n- **Queen’s Gambit Accepted**: Black takes the c-pawn.\n- **Queen’s Gambit Declined**: Black doesn’t take it.\n\nIf you want, I can also explain the idea behind it in simple terms or show a few common move lines. Anything else you want to explore?'

In [111]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess queen's gambit opening"}
iteration #2...
function_call: search {"query":"queen's gambit meaning chess opening course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen’s gambit,” so it looks off-topic for this course help desk.

If you meant something course-related, feel free to rephrase it and I’ll check the FAQ again. Any other areas you want to explore?


'I couldn’t find a course FAQ entry for “queen’s gambit,” so it looks off-topic for this course help desk.\n\nIf you meant something course-related, feel free to rephrase it and I’ll check the FAQ again. Any other areas you want to explore?'

### ToyAIKit

In [5]:
# !uv add toyaikit

In [15]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

import json

In [16]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [27]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [28]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [29]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [30]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [31]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [32]:
result.cost

CostInfo(input_cost=Decimal('0.000465'), output_cost=Decimal('0.0018675'), total_cost=Decimal('0.0023325'))

In [33]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama locally run local install Ollama FAQ"}', call_id='call_1Eu9n0O9pV8ccKqGVdmn6orW', name='search', type='function_call', id='fc_05af426e9bd66f20006a37db8c68dc8198bed7bade0d38bc82', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_1Eu9n0O9pV8ccKqGVdmn6orW',
  'output': '{\n  "er

In [34]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [35]:
runner.run()
# type "stop" to exit

You: Can I still join the course?


-> Response received


-> Response received


-> Response received


You: stop


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None), EasyInputMessage(content='Can I still join the course?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"join course enrollment late registration can I still join the course"}', call_id='call_XKlwIksCe0W2RsK6RAXQlikG', name='search', type='function_call', id='fc_0dbb981372bd6492006a37dcb0b98481988a2924b7f2aea369', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course